# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading, reviewing, and analyzing a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, hosted at [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

The dataset includes survey data on mental health among residents of Kilifi County, Kenya, with demographic details and psychological scores such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Access and print the main metadata info
metadata = dataset.metadata
print(metadata.name + ': ' + metadata.description)

## 2. Data Overview
Review available record sets, fields, and columns along with their `@id`.

All entities will be referenced by their `@id`.

Let's enumerate the record sets, fields, and columns in the dataset.

In [ ]:
# Explore Croissant metadata structure

# Access the list of record sets by @id
record_sets = []
if hasattr(metadata, 'recordSet'):
    if isinstance(metadata.recordSet, list):
        record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in metadata.recordSet]
    elif isinstance(metadata.recordSet, dict):
        record_sets = [metadata.recordSet['@id']]

if not record_sets:
    print('No record sets found in metadata. Check the dataset structure or schema.')
else:
    print('Record Sets in dataset:')
    for rs_id in record_sets:
        print('  -', rs_id)
        # Show fields in record set
        rs_obj = dataset.metadata_by_id(rs_id)
        if hasattr(rs_obj, 'field'):
            fields = rs_obj.field
            if isinstance(fields, list):
                for fld in fields:
                    fld_id = fld['@id'] if isinstance(fld, dict) else fld
                    print('    * Field @id:', fld_id)
            else:
                print('    * Field @id:', fields['@id'] if isinstance(fields, dict) else fields)
        # Show columns
        if hasattr(rs_obj, 'column'):
            columns = rs_obj.column
            if isinstance(columns, list):
                for col in columns:
                    col_id = col['@id'] if isinstance(col, dict) else col
                    print('      - Column @id:', col_id)
            else:
                print('      - Column @id:', columns['@id'] if isinstance(columns, dict) else columns)
        print()

## 3. Data Extraction
Load data from the available record sets into DataFrames for further analysis.

We use the record set and field/column `@id` discovered in the overview.

In [ ]:
# If record sets not found, check the typical structure
if not record_sets:
    # Attempt to infer main record set from typical Croissant Dataset structure
    # This is a fallback and may require manual inspection
    record_sets = []
    if hasattr(dataset, 'record_sets'):
        record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in dataset.record_sets]
    if not record_sets:
        raise RuntimeError('No record sets discovered. Please check schema definitions.')

# Prepare to collect data from each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id} columns:")
        print(df.columns.tolist())

# Display the first few rows from the first record set
if dataframes:
    first_rs_id = next(iter(dataframes))
    dataframes[first_rs_id].head()
else:
    print('No records loaded for any record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping records.

All columns and fields are referenced by their `@id`.

In [ ]:
import numpy as np

# Choose a record set and numeric field that exist, using discovered @id(s)
if dataframes:
    # Use the first record set and attempt to find relevant numeric fields
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Available columns for EDA in {record_set_id}:")
    print(df.columns.tolist())

    # Example: Use GAD-7, PHQ-9, or PSQ scores (replace with actual column @id if available)
    # These are placeholders; please adjust to actual @id column names discovered above.
    numeric_fields = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = 10
        # Filter for values above threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Group by available demographic fields
        # Example: 'gender', 'level_of_education', 'age', using @id
        group_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'age' in col.lower()]
        if group_fields:
            group_field_id = group_fields[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped data by {group_field_id}:")
                print(grouped_df.head())
    else:
        print('No numeric fields found for analysis.')

else:
    print('No DataFrames loaded; cannot perform EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships with demographic groups.

*Plots use `matplotlib` and columns referenced by their `@id`.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of the numeric field distribution
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    # Use the same numeric_field_id and group_field_id as above
    numeric_fields = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
    group_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'age' in col.lower()]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id} scores")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        # If group_field available, show boxplot
        if group_fields:
            group_field_id = group_fields[0]
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.title(f"{numeric_field_id} scores by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
    else:
        print('No numeric fields found for plotting.')
else:
    print('No DataFrames loaded; cannot plot.')

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to:
- Load FAIR² survey data on mental health from Kilifi County, Kenya using the Croissant schema.
- Discover and reference entities strictly by their `@id`.
- Extract and explore survey data, filter and normalize scores such as GAD-7 and PHQ-9 using their column `@id`.
- Visualize field distributions and demographic relationships.

The FAIR² Mental Health dataset provides a resource for data-driven insights into mental health indicators in Kilifi, Kenya, and demonstrates AI-ready, FAIR-compliant data standards for Africa.
